In [2]:
import torch 
import torch.nn as nn

# Get the data

Here I'm gonna simulate an embedding matrix already ready

(batch_size, tokens or seq_len, d_model)

In [4]:
x = torch.rand(2, 5, 10)
y = torch.rand(2, 5, 10)

# BTC (Batch, Tokens, Channels)

$B$ (Batch Size): How many sentences are you processing together (e.g. 32).

$T$ (Sequence Length / seq_len): The sentence size in tokens (e.g. 512).

$C$ (Channels / d_model): The dimension of the embedding (e.g. 768).

> PyTorch Golden Rule: Virtually all layers of your Transformer (Embeddings, FFN, LayerNorm) will operate in the last dimension ($C$). These process batch $B$ and sequence $T$ in parallel independently.

In [5]:
B,T,C = x.shape

# Transformer Components in PyTorch
 
---
 
## Hierarchy
 
```
nn.Transformer
└── nn.TransformerEncoder / nn.TransformerDecoder
      └── nn.TransformerEncoderLayer / nn.TransformerDecoderLayer
            └── nn.MultiheadAttention
                  └── F.scaled_dot_product_attention
```
 
Each level involves the previous one. The lower, the more control — and the more manual code.
 
---

# 1. `nn.Transformer`
 
Complete encoder-decoder model. Used for **seq2seq** tasks where input and output are distinct sequences (translation, summarization).
 
```python
model = nn.Transformer(
    d_model=512, # dimension of embeddings
    nhead=8, # number of attention heads
    num_encoder_layers=6, # encoder blocks
    num_decoder_layers=6, # decoder blocks
    dim_feedforward=2048, # internal dimension of FFN
    dropout=0.1,
    activation='relu', # 'relu' or 'gelu'
    batch_first=True, # (batch, seq, d) — always use True
)
 
out = model(src, tgt) # need both sequences
```
 
**Main Hyperparameters:**
 
| Parameter | What it does |
|---|---|
| 'd_model' | Size of embeddings across the model |
| 'nhead' | Number of heads — 'd_model' must be divisible by 'nhead' |
| 'num_encoder_layers' | How many blocks in the encoder |
| 'num_decoder_layers' | How many blocks in the decoder |
| 'dim_feedforward' | FFN internal size (usually '4 × d_model') |
| 'dropout' | Regularization |
| 'batch_first' | Sets dim order — always use 'True' |
 
**Returns:** '(batch, tgt_seq_len, d_model)' — decoder output.
 
**When to use:** Translation, summarization, any task with distinct input and output. In practice, little used — prefer T5/BART by HuggingFace.

In [52]:
transformer = nn.Transformer(d_model=10, nhead=10, num_decoder_layers=1, num_encoder_layers=1,  batch_first=True, dim_feedforward=20)
output = transformer(src=x, tgt= y)

print(transformer)
print(f'Output: {output}')

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=True)
        )
        (linear1): Linear(in_features=10, out_features=20, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=20, out_features=10, bias=True)
        (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0): TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=Tru

# 2. `nn.TransformerEncoderLayer` / `nn.TransformerDecoderLayer` (Layers)
 
A single transformer block. It's the repeatable unit — what the 'TransformerEncoder' stacks N times.

```python
# Encoder Layer — a complete block
layer = nn.TransformerEncoderLayer(
    d_model=512,
    nhead=8,
    dim_feedforward=2048,
    dropout=0.1,
    activation='relu', # 'relu', 'gelu' or custom function
    batch_first=True,
    norm_first=False, # False = Post-LN, True = Pre-LN (more stable)
)
 
out = layer(x) # (batch, seq_len, d_model) → (batch, seq_len, d_model)
```
 
**What happens internally:**
 
```
Post-LN(norm_first=False): Pre-LN(norm_first=True):
x → SelfAttn → Add → Norm x → Norm → SelfAttn → Add
  → FFN → Add → Norm → Norm → FFN → Add
```
 
**Extra Hyperparameters:**
 
| Parameter | What it does |
|---|---|
| 'norm_first' | 'True' = Pre-LN (more stable in training) |
| `activation` | FFN activation function |
 
**Returns:** same shape as input '(batch, seq_len, d_model)'.
 
**When to use:** Rapid prototyping, learning, baselines. When you don't need to customize the inside of the block.
 
---

In [26]:
encoder_layer = nn.TransformerEncoderLayer(d_model=C, nhead=1, dim_feedforward=C * 4, bias= False, batch_first=True)
output = encoder_layer(x)
print(encoder_layer)
print(f'Output: {output}')

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=False)
  )
  (linear1): Linear(in_features=10, out_features=40, bias=False)
  (dropout): Dropout(p=0.1, inplace=False)
  (linear2): Linear(in_features=40, out_features=10, bias=False)
  (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (dropout2): Dropout(p=0.1, inplace=False)
)
Output: tensor([[[ 0.9290,  1.3853,  1.0537,  0.1227, -0.5247, -0.4826,  0.9055,
          -1.2537, -1.7433, -0.3918],
         [ 2.3997,  0.5023,  0.2749, -0.3923, -0.1111, -0.2149, -1.4642,
           0.4866, -1.0751, -0.4060],
         [ 2.5661,  0.6319, -0.6539,  0.5811, -0.5738, -0.7956, -0.7671,
           0.1576, -0.6675, -0.4788],
         [ 2.6481,  0.3069,  0.3438, -0.6528, -0.4401,  0.0074, -1.3264,
          -0.1428, -0.5942, -0.1499],


# 3. `nn.TransformerEncoder` (Blocks)
 
Stack N 'EncoderLayer's. Does not add new logic — just repeats the block and applies optional final LayerNorm.

```python
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
 
encoder = nn.TransformerEncoder(
    encoder_layer=encoder_layer,
    num_layers=6, # how many blocks to stack
    norm=None, # Optional final LayerNorm
    enable_nested_tensor=True, # internal optimization
)
 
out = encoder(
    src=x,
    mask=None, # attention mask (seq, seq)
    src_key_padding_mask=None, # padding mask (batch, seq)
)
```
 
**Hyperparameters:**
 
| Parameter | What it does |
|---|---|
| `num_layers` | How many blocks to stack |
| 'norm' | LayerNorm applied after all blocks |
| 'src_key_padding_mask' | Mask padding tokens in batch |
 
**Returns:** '(batch, seq_len, d_model)' — contextualized embeddings.
 
**When to use:** Encoder-only tasks — classification, NER, embeddings, BERT-like.
 
---

In [29]:
encoder_block = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=10, enable_nested_tensor=False)
output = encoder_block(x)
print(encoder_block)
print(f'Output: {output}')

TransformerEncoder(
  (layers): ModuleList(
    (0-9): 10 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=False)
      )
      (linear1): Linear(in_features=10, out_features=40, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=40, out_features=10, bias=False)
      (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
    )
  )
)
Output: tensor([[[ 2.6165,  0.6171,  0.5666, -0.8348, -0.5636, -0.6483, -0.2835,
          -0.6530, -0.7055, -0.1115],
         [ 2.3236,  1.1070,  0.5736, -0.8878, -0.9264, -0.4518,  0.0744,
          -0.7704, -0.6846, -0.3577],
         [ 2.5851,  0.7960,  0.5719, -0.7813, -0.4736, -0.5884, -0.4160,
          -0.5365, -0.7289, -0.4284],
 

# 4. `F.scaled_dot_product_attention` (SDPA)
 
The pure operation of attention. No trainable parameters — just the math:
 
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
 
```python
import torch.nn.functional as F
 
out = F.scaled_dot_product_attention(
    query, # (batch, heads, seq, head_dim)
    key, # (batch, heads, seq, head_dim)
    value, # (batch, heads, seq, head_dim)
    attn_mask=None, # custom mask
    dropout_p=0.0, # dropout in attention weights
    is_causal=False, # automatic causal mask
    scale=None, # overwrite the default 1/√d
)
```
 
**Parameters:**
 
| Parameter | What it does |
|---|---|
| 'is_causal' | Applies triangular mask — token only sees the past |
| 'attn_mask' | Custom mask (boolean or float with '-inf') |
| 'dropout_p' | Dropout applied to weights after softmax |
| 'scale' | Scale factor — default is '1/√head_dim' |
 
**Returns:** only the output '(batch, heads, seq, head_dim)' — without attention weights.
 
**Automatically dispatches to FlashAttention** if GPU supports — without any extra configuration.
 
**When to use:** Custom implementations of LLMs, research, production. Base of LLaMA, Mistral, GPT-NeoX and all modern models.
 
---

In [49]:
import torch.nn.functional as F

q, k, v = torch.rand(5,10), torch.rand(5,10), torch.rand(5,10)
sdpa = F.scaled_dot_product_attention(query=q, key=k, value=v)

print(f'Output of SDPA:\n{sdpa}')


Output of SDPA:
tensor([[0.7452, 0.6380, 0.5967, 0.5010, 0.6118, 0.4405, 0.7294, 0.4044, 0.4818,
         0.3336],
        [0.7592, 0.6271, 0.5951, 0.4744, 0.6100, 0.4508, 0.7196, 0.3834, 0.4793,
         0.3314],
        [0.7525, 0.6356, 0.5908, 0.5038, 0.6126, 0.4326, 0.7229, 0.4047, 0.4857,
         0.3280],
        [0.7337, 0.6375, 0.6015, 0.5107, 0.6102, 0.4381, 0.7357, 0.4173, 0.4802,
         0.3344],
        [0.7448, 0.6322, 0.5983, 0.4976, 0.6123, 0.4389, 0.7252, 0.4060, 0.4822,
         0.3304]])


 
## 5. `nn.MultiheadAttention` (MHA)
 
Attention with multiple heads and trainable projections (Wq, Wk, Wv, Wo). Internally uses 'F.scaled_dot_product_attention'.
 
```python
mha = nn.MultiheadAttention(
    embed_dim=512, # dimension of embeddings
    num_heads=8, # number of heads
    dropout=0.0,
    bias=True, # bias in projections
    kdim=None, # dim das keys (default = embed_dim)
    vdim=None, # dim of values (default = embed_dim)
    batch_first=True, # always True
)

## Self-attention
attn_out, attn_weights = mha(query=x, key=x, value=x)
 
## Cross-attention
attn_out, attn_weights = mha(query=tgt, key=memory, value=memory)
 
## With causal mask
attn_out, attn_weights = mha(query=x, key=x, value=x, is_causal=True)
```
 
**Hyperparameters:**
 
| Parameter | What it does |
|---|---|
| 'embed_dim' | Input and output dimension |
| 'num_heads' | Number of heads — 'embed_dim' must be divisible |
| 'kdim' / 'vdim' | Allows K and V with dimension different from Q (cross-modal) |
| 'bias' | Bias in linear projections — modern models use 'False' |
 
**Returns:** tuple '(attn_output, attn_weights)':
- 'attn_output': '(batch, seq_len, embed_dim)' — post-attention embeddings
- 'attn_weights': '(batch, seq_len, seq_len)' — average of the head weights
**When to use:** Custom Cross-attention, when you need to inspect attention weights, or when 'TransformerEncoderLayer' doesn't cover the case.
 

In [47]:
mha = nn.MultiheadAttention(embed_dim=10, num_heads=10)
output, weights = mha(query=q, key=k, value=v)

print(f'Output:\n{output}') # The output of attention, this values goes to ffn
print(f'Weights:\n{weights}') # The attention matrix

Output:
tensor([[-0.1006, -0.3641, -0.4183, -0.3368, -0.0932, -0.1439, -0.2106, -0.2029,
         -0.4061, -0.1382],
        [-0.1031, -0.3631, -0.4186, -0.3343, -0.0964, -0.1476, -0.2073, -0.2007,
         -0.4061, -0.1368],
        [-0.1016, -0.3637, -0.4197, -0.3393, -0.0947, -0.1406, -0.2117, -0.2036,
         -0.4041, -0.1402],
        [-0.1005, -0.3669, -0.4209, -0.3401, -0.0944, -0.1433, -0.2117, -0.2046,
         -0.4028, -0.1382],
        [-0.1004, -0.3702, -0.4228, -0.3421, -0.0949, -0.1449, -0.2127, -0.2063,
         -0.4029, -0.1381]], grad_fn=<SqueezeBackward1>)
Weights:
tensor([[0.2024, 0.1954, 0.2068, 0.1861, 0.2093],
        [0.2007, 0.1966, 0.2031, 0.1926, 0.2070],
        [0.2061, 0.1875, 0.2097, 0.1840, 0.2126],
        [0.2063, 0.1877, 0.2061, 0.1874, 0.2125],
        [0.2072, 0.1868, 0.2064, 0.1874, 0.2122]], grad_fn=<SqueezeBackward1>)


---
 
## General comparison
 
| Component | Trainable parameters | Returns | When to use |
|---|---|---|---|
| 'F.sdpa' | No | output | LLMs, production, research |
| 'nn.MHA' | Yes (Wq,Wk,Wv,Wo) | output + weights | cross-attention,debug |
| 'EncoderLayer' | Yes (MHA + FFN) | output | prototyping, baselines |
| 'TransformerEncoder' | Yes (N × Layer) | output | full encoder-only |
| 'nn.Transformer' | Yes (encoder + decoder) | decoder output | seq2seq |
---
 
## General rule
 > is_causal can be a key difference here, cuz action like a mask

```
Learning / prototyping → nn.TransformerEncoderLayer
Custom Cross-attention → nn.MultiheadAttention
LLM / production → F.scaled_dot_product_attention
Full Seq2seq → nn.Transformer (or T5/BART via HuggingFace)
```


My own resume: 
Review of functions in Torch:

	nn.Transformers = Function that defines everything from block to layers returns the final array (post attention, add and norm, and ffn)

	nn.TransformersEncoder/Decoder = Function that defines a block, we need layer function to use returns the final array (post attention, add and norm and ffn), 

	nn.TransformersEncoder/DecoderLayer = Function that Defines a layer to join into blocks, returns the final array (post attention, add and norm and ffn)
		- Decoder has cross-attention, that is, it has decoder + encoder (that's why other doq esse are more used)

	nn.MultiHeadAttention= Function that receives the QKV matrices and calculates the attention, returns the post-attention embedding matrix and the attention matrix
		- We use it when we want to have attention results

	F.scaled_dot_product_attention = Function that receives the QKV matrices and calculates the attention, returns the embedding matrix after attention
		- We use it when we want to have the results of attention and attention matrix

	nn.Linear = Trainable weight matrix
		- We use it as the basis for the model weights

Hyperparameter review

	vocab_size = Size of the rows of embedding matrix (sets the amount of tokens our embedding matrix has)

	seq_len/tokens = Sets how many tokens in the current sample

	d_model(embedding dimensions) = defines how many columns, our embedding matrix has (number of features)

	dim_feedforward = Sets the number of neurons we will have in the ffn

	dropout = Sets the dropout of the ffn

	heads = Sets the number of attention heads we will have (note d_model / heads)

	blocks/layers = Sets the number of blocks we will have in our model

	is_causal = Defines whether our model will use mask or not

	batch_first = Sets the shape to be (B,T,C) or (B,H, T,C)
    
 	bias = Defines whether or not we will use bias in nn.Linear()